# 01 — Exploration, extraction de features et clustering exploratoire

**Mission BrainScanAI — *CurelyticsIA*** &nbsp;·&nbsp; *Option B*

Vous êtes Data Scientist junior en computer vision. Le projet R&D *BrainScanAI*
vise à automatiser la détection de tumeurs cérébrales sur des IRM. Le dataset
fourni par Clara Martin contient :

- **100 images fortement labellisées** (50 *normal* / 50 *cancer*) annotées par
  des radiologues partenaires ;
- **1 406 images non labellisées** issues du même flux d'acquisition.

Ce premier notebook couvre les **étapes 1 à 3** de la mission :

1. Chargement et exploration du jeu de radiographies.
2. Prétraitement et **extraction des features** via un modèle pré-entraîné.
3. **Analyse non supervisée** : réduction de dimension + clustering, comparaison
   de plusieurs algorithmes, choix d'une méthode et **labellisation faible** des
   1 406 images non annotées.

> ⚠️ **Règle métier non négociable** — les jeux *fortement* et *faiblement*
> labellisés ne sont jamais mélangés ; ils sont conservés dans deux structures
> séparées tout au long du projet.

## Définition du *done* (notebook 1)

| Critère | Cible |
|---|---|
| Inventaire complet du dataset (intégrité, résolution, modes) | ✅ |
| Outliers documentés et traités | ✅ |
| Features ResNet50 calculées pour les 1 506 images | ✅ |
| Au moins **4 algorithmes de clustering** testés | ✅ |
| ARI clustering (vs labels forts) ≥ 0.10 (mieux que le hasard) | objectif |
| Pseudo-labels « faibles » exportés en CSV pour le notebook 2 | ✅ |

L'erreur la plus coûteuse est le **faux négatif sur cancer** (passer à côté
d'une tumeur). Cette priorité métier oriente la sélection des métriques (recall
de la classe *cancer* avant accuracy globale) — détaillé dans le notebook 2.

## 1. Chargement et exploration du jeu de radiographies

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from curelyticsia.config import (
    CLASS_TO_INDEX,
    CLASSES,
    FIGURES_DIR,
    SEED,
    ClusteringConfig,
    FeatureConfig,
    ensure_dirs,
    set_global_seeds,
)

ensure_dirs()
set_global_seeds(SEED)

print(f"Project root : {ROOT}")
print(f"Classes      : {CLASSES} → {CLASS_TO_INDEX}")
print(f"Seed         : {SEED}")

In [ ]:
# Extraction du dataset (idempotent : ne ré-extrait pas si déjà présent).
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "extract_dataset.py")],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

In [ ]:
from curelyticsia.data.loader import (
    discover_images,
    records_to_dataframe,
    summarise_dataset,
)

records, corrupted = discover_images()
print(f"{len(records):>5} images valides")
print(f"{len(corrupted):>5} images corrompues écartées")
if corrupted:
    for p in corrupted[:5]:
        print("  -", p)

df = records_to_dataframe(records)
df.head()

### 1.1 Vue d'ensemble

In [ ]:
summary = summarise_dataset(records)
print("Total       :", summary["total"])
print("Par split   :", summary["by_split"])
print("Par label   :", summary["by_label"])
print("Modes       :", summary["modes"])
print("Résolutions :", dict(list(summary["resolutions"].items())[:5]))

**Observations attendues** (à confirmer à l'exécution) :

- 100 images étiquetées (50 *normal* + 50 *cancer*) dans `avec_labels/` et
  ~1 400 images non labellisées dans `sans_label/`.
- Toutes les images sont en JPEG 512×512 — la documentation fournie le précise
  et l'inventaire le confirme.
- Le mode dominant est `L` (niveaux de gris) ; il est explicitement converti
  en RGB par le pipeline de preprocessing avant ResNet (qui attend 3 canaux
  ImageNet).

### 1.2 Galeries d'exemples

In [ ]:
from curelyticsia.viz.plots import plot_image_grid

mask_cancer = (df["split"] == "labeled") & (df["label_name"] == "cancer")
mask_normal = (df["split"] == "labeled") & (df["label_name"] == "normal")
mask_unlab = df["split"] == "unlabeled"

cancer_paths = df.loc[mask_cancer, "path"].head(10).tolist()
normal_paths = df.loc[mask_normal, "path"].head(10).tolist()
unlab_paths = df.loc[mask_unlab, "path"].head(10).tolist()

_ = plot_image_grid(
    cancer_paths,
    titles=[f"cancer · {Path(p).name[:8]}" for p in cancer_paths],
    cols=5,
    save_path=FIGURES_DIR / "samples_cancer.png",
)

In [ ]:
_ = plot_image_grid(
    normal_paths,
    titles=[f"normal · {Path(p).name[:8]}" for p in normal_paths],
    cols=5,
    save_path=FIGURES_DIR / "samples_normal.png",
)

In [ ]:
_ = plot_image_grid(
    unlab_paths,
    titles=[f"? · {Path(p).name[:8]}" for p in unlab_paths],
    cols=5,
    save_path=FIGURES_DIR / "samples_unlabeled.png",
)

### 1.3 Statistiques de pixels et détection d'outliers

In [ ]:
from curelyticsia.data.loader import compute_pixel_stats, detect_outlier_ids
from curelyticsia.viz.plots import plot_pixel_stats

stats = compute_pixel_stats(records, sample_size=None)  # toutes les images
print(stats.describe())

_ = plot_pixel_stats(stats, save_path=FIGURES_DIR / "pixel_stats.png")

In [ ]:
outlier_ids = detect_outlier_ids(stats, z_threshold=4.0)
print(f"{len(outlier_ids)} outliers détectés (|z| > 4 sur mean ou std)")

# Filtre éventuel : on les conserve dans cette mission (peu nombreux et
# potentiellement informatifs en imagerie médicale) mais on documente leur
# présence pour l'étape clustering.
records_clean = [r for r in records if r.image_id not in set(outlier_ids)]
print(f"{len(records_clean)} images conservées après filtrage soft")

**Choix de traitement des outliers** : on adopte un seuil très conservateur
(*z* = 4) pour ne supprimer que des cas réellement aberrants (capteur défectueux
par exemple). En IRM cérébrale, des écarts d'intensité significatifs peuvent
correspondre à des tumeurs ; supprimer trop agressivement biaiserait la
détection. Le traitement reste documenté ici pour traçabilité.

## 2. Prétraitement et extraction de features ResNet50

**Stratégie** :

- Resize à 256, *center crop* à 224×224, normalisation **ImageNet** (mean / std).
- Backbone **ResNet50** pré-entraîné sur ImageNet, **toutes les couches gelées**
  (recommandation explicite de l'énoncé : *« geler les couches convolutionnelles »*).
- La tête de classification ImageNet est remplacée par `Identity` ; on récupère
  la sortie du *global average pool* — un vecteur de **2 048 dimensions** par
  image.
- Mise en cache au format **Parquet** : la prochaine exécution est instantanée.

In [ ]:
from curelyticsia.features.extractor import extract_features

cfg = FeatureConfig()
features, index_df = extract_features(records, cfg=cfg, use_cache=True, progress=True)
print("features.shape :", features.shape)
print(index_df["split"].value_counts().to_string())

In [ ]:
# Vérification que les sorties (les embeddings) sont bien exploitables.
print("Aucun NaN ?      ", not np.isnan(features).any())
print("Norme moyenne L2 :", float(np.linalg.norm(features, axis=1).mean()))
print("Variance par dim :", float(features.var(axis=0).mean()))

## 3. Analyse non supervisée

### 3.1 Standardisation et réduction de dimension

On centre-réduit les 2 048 dimensions, puis on conserve **95 % de la variance**
via PCA. La PCA accélère le clustering et stabilise t-SNE / UMAP.

In [ ]:
from curelyticsia.models.clustering import reduce_pca, standardise

features_std, scaler = standardise(features)
features_red, pca = reduce_pca(features_std, target_variance=0.95)
print(f"Dimensions PCA : {features_red.shape[1]} (variance cumulée ≥ 0.95)")
print(f"Première composante explique {pca.explained_variance_ratio_[0]:.2%} de la variance")

### 3.2 Visualisation 2D (t-SNE / UMAP)

In [ ]:
from curelyticsia.viz.plots import plot_2d_scatter, project_2d

labels_for_color = index_df["label_name"].fillna(value="").replace("", None).tolist()

coords_tsne = project_2d(features_red, method="tsne", seed=SEED)
_ = plot_2d_scatter(
    coords_tsne,
    labels=labels_for_color,
    title="t-SNE des embeddings ResNet50 — colorés par label connu",
    save_path=FIGURES_DIR / "tsne_labels.png",
)

coords_umap = project_2d(features_red, method="umap", seed=SEED)
_ = plot_2d_scatter(
    coords_umap,
    labels=labels_for_color,
    title="UMAP des embeddings ResNet50 — colorés par label connu",
    save_path=FIGURES_DIR / "umap_labels.png",
)

### 3.3 Comparaison de plusieurs algorithmes de clustering

In [ ]:
from curelyticsia.models.clustering import (
    align_cluster_labels,
    assign_weak_labels,
    build_clustering_report,
    export_weak_labels,
    fit_agglomerative,
    fit_dbscan,
    fit_gmm,
    fit_kmeans,
)

truth = np.where(
    index_df["split"] == "labeled",
    index_df["label_name"].map(lambda v: CLASS_TO_INDEX.get(v) if v else np.nan),
    np.nan,
).astype(float)

results = [
    fit_kmeans(features_red, truth, n_clusters=2, seed=SEED),
    fit_agglomerative(features_red, truth, n_clusters=2, linkage="ward"),
    fit_agglomerative(features_red, truth, n_clusters=2, linkage="average"),
    fit_gmm(features_red, truth, n_components=2, seed=SEED),
    fit_dbscan(features_red, truth, eps=8.0, min_samples=10),
]

report = build_clustering_report(results)
report

**Lecture** :

- *Silhouette* (↑) et *Calinski-Harabasz* (↑) mesurent la séparation interne
  des clusters. *Davies-Bouldin* (↓) la dispersion ; on cherche donc la valeur
  la plus basse.
- L'**ARI** est l'arbitre : il compare la partition obtenue à la *vérité
  terrain* sur les 100 images étiquetées. ARI ∈ [-1, 1] ; 0 = hasard.
- DBSCAN peut produire un nombre arbitraire de clusters (ou 0). Sur des
  embeddings 2048d, il est rarement compétitif sans tuning approfondi : on le
  conserve néanmoins comme baseline pour tracer le résultat.

### 3.4 Sélection du meilleur clustering et alignement des labels

In [ ]:
candidates = [r for r in results if r.ari_vs_truth is not None]
best = max(candidates, key=lambda r: r.ari_vs_truth)
print(f"Méthode retenue : {best.name}")
print(f"ARI vs vérité   : {best.ari_vs_truth:.4f}")
print(f"Silhouette      : {best.silhouette}")
print(f"Davies-Bouldin  : {best.davies_bouldin}")
print(f"Calinski-Hara.  : {best.calinski_harabasz}")

In [ ]:
aligned = align_cluster_labels(best.labels, truth)

# Visualisation des clusters alignés sur la projection t-SNE.
labels_for_color_aligned = [None if v is None else v for v in (
    pd.Series(aligned).map({0: "normal", 1: "cancer", -1: "noise"}).tolist()
)]
_ = plot_2d_scatter(
    coords_tsne,
    labels=labels_for_color_aligned,
    title=f"Clusters alignés ({best.name}) — projection t-SNE",
    save_path=FIGURES_DIR / "tsne_clusters_aligned.png",
)

### 3.5 Pseudo-labels « faibles » sur les images non annotées

In [ ]:
weak = assign_weak_labels(index_df, aligned)
print(f"{len(weak)} pseudo-labels exportables")
print(weak["weak_label_name"].value_counts())

out = export_weak_labels(weak, ClusteringConfig())
print(f"\nExport : {out}")

**Vérification de la séparation des jeux** : `weak` ne contient **que** des
images du split `unlabeled` ; aucune image fortement labellisée n'apparaît.
La règle métier *« ne jamais mélanger faible et fort »* est respectée.

## 4. Synthèse

- Inventaire propre du dataset (modes, résolutions, intégrité, outliers).
- Embeddings ResNet50 calculés et mis en cache → réutilisables sans recalcul.
- Quatre algorithmes de clustering comparés (K-Means, Agglomerative ×2, GMM,
  DBSCAN). La méthode retenue est celle qui maximise l'ARI face aux 100 labels
  forts.
- Les pseudo-labels « faibles » sont exportés vers
  `data/processed/weak_labels.csv` ; ils alimentent le **notebook 02** où l'on
  entraîne et compare un CNN supervisé pur vs un CNN semi-supervisé.

L'analyse pour le **passage à l'échelle (4 M images / 5 000 €)** est traitée
dans le support de présentation.